# CASPER Quick test run

In [1]:
import torch, os, string
from transformers import AutoModelForMaskedLM, AutoTokenizer
from splade.models.transformer_rep import CASPERv2
from collections import Counter

/home/lamdo/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [3]:
# set the dir for trained weights
model_type_or_dir = "/scratch/lamdo/casperv2/checkpoints/test9/debug/checkpoint/model"  #"lamdo/distilbert-base-uncased-phrase-30kaddedphrasesfroms2orc-mlm-70000steps" 
concept_level_indices_path = "/scratch/lamdo/casperv2/concept_level_indices/17Oct2025.json"

In [4]:
# loading model and tokenizer

model = CASPERv2(model_type_or_dir, agg="max", original_bert_vocab_size=30522, concept_level_indices_path=concept_level_indices_path)
# model = Splade(model_type_or_dir, agg = "max")
model.eval()
tokenizer = AutoTokenizer.from_pretrained(model_type_or_dir)
reverse_voc = {v: k for k, v in tokenizer.vocab.items()}

len(reverse_voc)

59419

In [5]:
def encode__(tokens, is_q, model):
    out = model.encode_(tokens, is_q)["logits"][:,1:-1]  # shape (bs, pad_len, voc_size)
    if model.agg == "sum":
        raise NotImplementedError
    else:
        out_tokens = out[..., :model.original_bert_vocab_size] # shape (bs, pad_len, original_bert_vocab_size)
        out_phrases = out[..., model.original_bert_vocab_size:] # shape (bs, pad_len, vocab_size - original_bert_vocab_size)
        values_tokens, _ = torch.max(torch.log(1 + torch.relu(out_tokens)) * tokens["attention_mask"][:,1:-1].unsqueeze(-1), dim=1) # shape (bs, original_bert_vocab_size)
        values_phrases = torch.sum(torch.log(1 + torch.relu(out_phrases)) * tokens["attention_mask"][:,1:-1].unsqueeze(-1), dim=1) # shape (bs, vocab_size - original_bert_vocab_size)

        values = torch.cat([values_tokens, values_phrases], dim = -1)
        return values
        # 0 masking also works with max because all activations are positive

In [9]:
doc = """ColBERTv2: Effective and Efficient Retrieval via Lightweight Late Interaction. Neural information retrieval (IR) has greatly advanced search and other knowledge-intensive language tasks. While many neural IR methods encode queries and documents into single-vector representations, late interaction models produce multi-vector representations at the granularity of each token and decompose relevance modeling into scalable token-level computations. This decomposition has been shown to make late interaction more effective, but it inflates the space footprint of these models by an order of magnitude. In this work, we introduce ColBERTv2, a retriever that couples an aggressive residual compression mechanism with a denoised supervision strategy to simultaneously improve the quality and space footprint of late interaction. We evaluate ColBERTv2 across a wide range of benchmarks, establishing state-of-the-art quality within and outside the training domain while reducing the space footprint of late interaction models by 6--10×."""

# doc = """Supplementing Remote Sensing of Ice: Deep Learning-Based Image Segmentation System for Automatic Detection and Localization of Sea-ice Formations From Close-Range Optical Images. This paper presents a three-stage approach for the automated analysis of close-range optical images containing ice objects. The proposed system is based on an ensemble of deep learning models and conditional random field postprocessing. The following surface ice formations were considered: Icebergs, Deformed ice, Level ice, Broken ice, Ice floes, Floebergs, Floebits, Pancake ice, and Brash ice. Additionally, five non-surface ice categories were considered: Sky, Open water, Shore, Underwater ice, and Melt ponds. To find input parameters for the approach, the performance of 12 different neural network architectures was explored and evaluated using a 5-fold cross-validation scheme. The best performance was achieved using an ensemble of models having pyramid pooling layers (PSPNet, PSPDenseNet, DeepLabV3+, and UPerNet) and convolutional conditional random field postprocessing with a mean intersection over union score of 0.799, and this outperformed the best single-model approach. The results of this study show that when per-class performance was considered, the Sky was the easiest class to predict, followed by Deformed ice and Open water. Melt pond was the most challenging class to predict. Furthermore, we have extensively explored the strengths and weaknesses of our approach and, in the process, discovered the types of scenes that pose a more significant challenge to the underlying neural networks. When coupled with optical sensors and AIS, the proposed approach can serve as a supplementary source of large-scale ‘ground truth’ data for validation of satellite-based sea-ice products. We have provided an implementation of the approach at https://github.com/panchinabil/sea_ice_segmentation ."""

# doc = """Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks. Large pre-trained language models have been shown to store factual knowledge in their parameters, and achieve state-of-the-art results when fine-tuned on downstream NLP tasks. However, their ability to access and precisely manipulate knowledge is still limited, and hence on knowledge-intensive tasks, their performance lags behind task-specific architectures. Additionally, providing provenance for their decisions and updating their world knowledge remain open research problems. Pre-trained models with a differentiable access mechanism to explicit non-parametric memory can overcome this issue, but have so far been only investigated for extractive downstream tasks. We explore a general-purpose fine-tuning recipe for retrieval-augmented generation (RAG) -- models which combine pre-trained parametric and non-parametric memory for language generation. We introduce RAG models where the parametric memory is a pre-trained seq2seq model and the non-parametric memory is a dense vector index of Wikipedia, accessed with a pre-trained neural retriever. We compare two RAG formulations, one which conditions on the same retrieved passages across the whole generated sequence, the other can use different passages per token. We fine-tune and evaluate our models on a wide range of knowledge-intensive NLP tasks and set the state-of-the-art on three open domain QA tasks, outperforming parametric seq2seq models and task-specific retrieve-and-extract architectures. For language generation tasks, we find that RAG models generate more specific, diverse and factual language than a state-of-the-art parametric-only seq2seq baseline."""

# doc = "Measurements of Macroscopic Quantum Tunneling out of the Zero-Voltage State of a Current-Biased Josephson Junction. The escape rate of an underdamped (𝑄≈30), current-biased Josephson junction from the zero-voltage state has been measured. The relevant parameters of the junction were determined in situ in the thermal regime from the dependence of the escape rate on bias current and from resonant activation in the presence of microwaves. At low temperatures, the escape rate became independent of temperature with a value that, with no adjustable parameters, was in excellent agreement with the zero-temperature prediction for macroscopic quantum tunneling."

# doc = "Unsupervised Dense Information Retrieval with Contrastive Learning. Recently, information retrieval has seen the emergence of dense retrievers, using neural networks, as an alternative to classical sparse methods based on term-frequency. These models have obtained state-of-the-art results on datasets and tasks where large training sets are available. However, they do not transfer well to new applications with no training data, and are outperformed by unsupervised term-frequency methods such as BM25. In this work, we explore the limits of contrastive learning as a way to train unsupervised dense retrievers and show that it leads to strong performance in various retrieval settings. On the BEIR benchmark our unsupervised model outperforms BM25 on 11 out of 15 datasets for the Recall@100. When used as pre-training before fine-tuning, either on a few thousands in-domain examples or on the large MS~MARCO dataset, our contrastive model leads to improvements on the BEIR benchmark. Finally, we evaluate our approach for multi-lingual retrieval, where training data is even scarcer than for English, and show that our approach leads to strong unsupervised performance. Our model also exhibits strong cross-lingual transfer when fine-tuned on supervised English data only and evaluated on low resources language such as Swahili. We show that our unsupervised models can perform cross-lingual retrieval between different scripts, such as retrieving English documents from Arabic queries, which would not be possible with term matching methods."

doc = "ColPali: Efficient Document Retrieval with Vision Language Models. Documents are visually rich structures that convey information through text, but also figures, page layouts, tables, or even fonts. Since modern retrieval systems mainly rely on the textual information they extract from document pages to index documents -often through lengthy and brittle processes-, they struggle to exploit key visual cues efficiently. This limits their capabilities in many practical document retrieval applications such as Retrieval Augmented Generation (RAG). To benchmark current systems on visually rich document retrieval, we introduce the Visual Document Retrieval Benchmark ViDoRe, composed of various page-level retrieval tasks spanning multiple domains, languages, and practical settings. The inherent complexity and performance shortcomings of modern systems motivate a new concept; doing document retrieval by directly embedding the images of the document pages. We release ColPali, a Vision Language Model trained to produce high-quality multi-vector embeddings from images of document pages. Combined with a late interaction matching mechanism, ColPali largely outperforms modern document retrieval pipelines while being drastically simpler, faster and end-to-end trainable"

# doc = "scientific document retrieval"

# doc = "A novel mutation in CD83 results in the development of a unique population of CD4+ T cells. Using a mouse mutagenesis screen, we have identified CD83 as being critical for the development of CD4(+) T cells and for their function postactivation. CD11c(+) dendritic cells develop and function normally in mice with a mutated CD83 gene but CD4(+) T cell development is substantially reduced. Additionally, we now show that those CD4(+) cells that develop in a CD83 mutant animal fail to respond normally following allogeneic stimulation. This is at least in part due to an altered cytokine expression pattern characterized by an increased production of IL-4 and IL-10 and diminished IL-2 production. Thus, in addition to its role in selection of CD4(+) T cells, absence of CD83 results in the generation of cells with an altered activation and cytokine profile."

# doc = " Improving Energy Conversion Efficiency of Ion-Driven Artificial Muscles Based on Carbon Nanotube Yarn. While artificial muscles provide giant work and power densities compared to natural muscles, their reported energy conversion efficiencies have so far been low. We here demonstrate a tension optimization process (TOP) for fabricating coiled carbon nanotube artificial muscles having record efficiencies. These TOP muscles were made by applying about 20 times higher tensile stress during pre-coiling twist insertion than the tensile stress applied during coiling, resulting in high twist density and high spring index. The TOP muscles driven by the tetrabutylammonium cation provide 6.1 J/g contractile work, which is ∼152 times the maximum capability of human skeletal muscles, and 13.1 % contractile energy efficiency. In addition, the contractile energy efficiency of the TOP muscles driven by the bis(trifluoromethanesulfonyl)imide anion is maximized to 38.8 % by minimizing side redox reactions. In the case of full-cycle actuation, which considers the whole cycle of contraction and relaxation, we increased the full-cycle energy conversion efficiency of TOP muscles to 6.7 %, which is 4.5 times that previously reported for ion-driven artificial muscles."


# doc = "Collective Charge Excitations Studied by Electron Energy-Loss Spectroscopy. The dynamic charge susceptibility, χ(q, ω), is a fundamental observable of all materials, in one, two, and three dimensions, quantifying the collective charge modes and the ability of a material to screen charge, as well as its electronic compressibility. Here, we review the current state of efforts to measure the charge susceptibility of quantum materials using inelastic electron scattering, which historically has been called electron energy-loss spectroscopy (EELS). We focus on comparison between transmission (T-EELS) and reflection (R-EELS) geometries as applied to a selection of three-dimensional and quasi-two-dimensional conductors. Although a great deal is understood about simple metals, measurements of more strongly interacting and strange metals are currently conflicting, with different groups obtaining fundamentally contradictory results, emphasizing the importance of improved EELS measurements. Furthermore, current opportunities for improvement in EELS techniques are vast, with the most promising future development being in hemispherical and time-of-flight analyzers, as well as scanning transmission electron microscope instruments configured for high-momentum resolution. We conclude that, despite more than half a century of work, EELS techniques are currently still in their infancy."


# doc = "ERU-KG: Efficient Reference-aligned Unsupervised Keyphrase Generation. Unsupervised keyphrase prediction has gained growing interest in recent years. However, existing methods typically rely on heuristically defined importance scores, which may lead to inaccurate informativeness estimation. In addition, they lack consideration for time efficiency. To solve these problems, we propose ERU-KG, an unsupervised keyphrase generation (UKG) model that consists of an informativeness and a phraseness module. The former estimates the relevance of keyphrase candidates, while the latter generate those candidates. The informativeness module innovates by learning to model informativeness through references (e.g., queries, citation contexts, and titles) and at the term-level, thereby 1) capturing how the key concepts of documents are perceived in different contexts and 2) estimating informativeness of phrases more efficiently by aggregating term informativeness, removing the need for explicit modeling of the candidates. ERU-KG demonstrates its effectiveness on keyphrase generation benchmarks by outperforming unsupervised baselines and achieving on average 89% of the performance of a supervised model for top 10 predictions. Additionally, to highlight its practical utility, we evaluate the model on text retrieval tasks and show that keyphrases generated by ERU-KG are effective when employed as query and document expansions. Furthermore, inference speed tests reveal that ERU-KG is the fastest among baselines of similar model sizes. Finally, our proposed model can switch between keyphrase generation and extraction by adjusting hyperparameters, catering to diverse application requirements."


# doc = "DiffPhys: Differential Physics Augmentations for Enhanced Representations. Foundation Models (FMs) have revolutionized representation learning in IoT sensing applications. However, these models face a critical limitation: while their generalized representations excel at detection despite environmental distortions, they struggle to differentiate between fine-grained variations in these distortions—a capability essential for many IoT tasks like proximity assessment and dynamic tracking. Traditional augmentation approaches exacerbate this problem by focusing on invariance to these distortions, teaching models to ignore rather than distinguish meaningful environmental variations. To address these limitations, we introduce DiffPhys, a novel framework that fundamentally shifts how models learn from augmentations. DiffPhys incorporates two key innovations: (i) progressive physics-guided augmentations modeling environmental effects at varying intensities, and (ii) an ordinal consistency constraint structuring the embedding space to preserve physical relationships. To support this framework, we implement differentiable physics-guided augmentations that model progressive environmental effects like attenuation, Doppler shifts, and scattering. DiffPhys is designed as a pluggable module that seamlessly integrates into existing IoT-driven ML pipelines without requiring architectural changes to the underlying models. Evaluations across vehicle classification, speed estimation, and distance tracking tasks show that DiffPhys can enhance models to achieve up to 9% improvement in environmental differentiation tasks compared to standard versions."


# doc = "Efficient, selective sodium and lithium removal by faradaic deionization using symmetric sodium titanium vanadium phosphate intercalation electrodes. NASICON (sodium superionic conductor) materials are promising host compounds for the reversible capture of Na+ ions, finding prior application in batteries as solid-state electrolytes and cathodes/anodes. Given their affinity for Na+ ions, these materials can be used in Faradaic deionization (FDI) for the selective removal of sodium over other competing ions. Here, we investigate the selective removal of sodium over other alkali and alkaline-earth metal cations from aqueous electrolytes when using a NASICON-based mixed Ti–V phase as an intercalation electrode, namely, sodium titanium vanadium phosphate (NTVP). Galvanostatic cycling experiments in three-electrode cells with electrolytes containing Na+, K+, Mg2+, Ca2+, and Li+ reveal that only Na+ and Li+ can intercalate into the NTVP crystal structure, while other cations show capacitive response, leading to a material-intrinsic selectivity factor of 56 for Na+ over K+, Mg2+, and Ca2+. Furthermore, electrochemical titration experiments together with modeling show that an intercalation mechanism with a limited miscibility gap for Na+ in NTVP mitigates the state-of-charge gradients to which phase-separating intercalation electrodes are prone when operated under electrolyte flow. NTVP electrodes are then incorporated into an FDI cell with automated fluid recirculation to demonstrate up to 94% removal of sodium in streams with competing alkali/alkaline-earth cations with 10-fold higher concentration, showing process selectivity factors of 3–6 for Na+ over cations other than Li+. Decreasing the current density can improve selectivity up to 25% and reduce energy consumption by as much as ∼50%, depending on the competing ion. The results also indicate the utility of NTVP for selective lithium recovery."


# doc = "The persistent shadow of the supermassive black hole of M87. The Event Horizon Telescope (EHT) observation of M87∗ in 2018 has revealed a ring with a diameter that is consistent with the 2017 observation. The brightest part of the ring is shifted to the southwest from the southeast. In this paper, we provide theoretical interpretations for the multi-epoch EHT observations for M87∗ by comparing a new general relativistic magnetohydrodynamics model image library with the EHT observations for M87∗ in both 2017 and 2018. The model images include aligned and tilted accretion with parameterized thermal and nonthermal synchrotron emission properties. The 2018 observation again shows that the spin vector of the M87∗ supermassive black hole is pointed away from Earth. A shift of the brightest part of the ring during the multi-epoch observations can naturally be explained by the turbulent nature of black hole accretion, which is supported by the fact that the more turbulent retrograde models can explain the multi-epoch observations better than the prograde models. The EHT data are inconsistent with the tilted models in our model image library. Assuming that the black hole spin axis and its large-scale jet direction are roughly aligned, we expect the brightest part of the ring to be most commonly observed 90 deg clockwise from the forward jet. This prediction can be statistically tested through future observations."

# doc = "Wearable strain sensors based on thin graphite films for human activity monitoring Wearable health-monitoring devices have attracted increasing attention in disease diagnosis and health assessment. In many cases, such devices have been prepared by complicated multistep procedures which result in the waste of materials and require expensive facilities. In this study, we focused on pyrolytic graphite sheet (PGS), which is a low-cost, simple, and flexible material, used as wearable devices for monitoring human activity. We investigated wearable devices based on PGSs for the observation of elbow and finger motions. The thin graphite films were fabricated by cutting small films from PGSs. The wearable devices were then made from the thin graphite films assembled on a commercially available rubber glove. The human motions could be observed using the wearable devices. Therefore, these results suggested that the wearable devices based on thin graphite films may broaden their application in cost-effective wearable electronics for the observation of human activity."

# doc = "The role of spread through air spaces (STAS) in lung adenocar cinoma prognosis and therapeutic decision making."

# doc = "keyphrase generation"

# doc = "Thermally Conductive Reduced Graphene Oxide Thin Films for Extreme Temperature Sensors Reduced graphene oxide (RGO) films are promising in applications ranging from electronics to flexible sensors"

# doc = "Weighted Turán Theorems With Applications to Ramsey-Turán Type of Problems. We study extensions of Turán Theorem in edge-weighted settings. A particular case of interest is when constraints on the weight of an edge come from the order of the largest clique containing it. These problems are motivated by Ramsey-Turán type problems. Some of our proofs are based on the method of graph Lagrangians, while the other proofs use flag algebras. Using these results, we prove several new upper bounds on the Ramsey-Turán density of cliques. Other applications of our results are in a recent paper of Balogh, Chen, McCourt, and Murley."

In [10]:
# # now compute the document representation
# for punc in string.punctuation:
#     doc = doc.replace(punc, " ")

prefix = "" #"Published by department of [MASK] ."
_doc = prefix + doc
doc_tokens = tokenizer(doc, max_length = 256, return_tensors="pt")
with torch.no_grad():
    doc_rep = model(d_kwargs=doc_tokens)["d_rep"].squeeze()  # (sparse) doc rep in voc space, shape (30522,)
    # print(torch.sum(doc_rep))
    # doc_rep = encode__(doc_tokens, False, model).squeeze()
print(doc_rep.shape)
# get the number of non-zero dimensions in the rep:
col = torch.nonzero(doc_rep).squeeze().cpu().tolist()
print("number of actual dimensions: ", len(col))

# now let's inspect the bow representation:
weights = doc_rep[col].cpu().tolist()

print("input text:", doc)

for level in ["venue", "keyphrases", "tokens"][:]:
    d = {k: v for k, v in zip(col, weights) if k in model.concept_level_indices[level]}
    sorted_d = {k: v for k, v in sorted(d.items(), key=lambda item: item[1], reverse=True)}
    bow_rep = []

    for k, v in sorted_d.items():
        # print((reverse_voc[k], round(v, 2)))
        bow_rep.append((reverse_voc[k], round(v, 2)))

    print(f"level '{level}' (total: {len(bow_rep)})", bow_rep[:50])
# print("SPLADE BOW rep:\n", bow_rep)

torch.Size([59419])
number of actual dimensions:  255
input text: ColPali: Efficient Document Retrieval with Vision Language Models. Documents are visually rich structures that convey information through text, but also figures, page layouts, tables, or even fonts. Since modern retrieval systems mainly rely on the textual information they extract from document pages to index documents -often through lengthy and brittle processes-, they struggle to exploit key visual cues efficiently. This limits their capabilities in many practical document retrieval applications such as Retrieval Augmented Generation (RAG). To benchmark current systems on visually rich document retrieval, we introduce the Visual Document Retrieval Benchmark ViDoRe, composed of various page-level retrieval tasks spanning multiple domains, languages, and practical settings. The inherent complexity and performance shortcomings of modern systems motivate a new concept; doing document retrieval by directly embedding the ima

In [8]:
def encode_custom(tokens, model, is_q = False):
    out = model.encode_(tokens, is_q)["logits"]  # shape (bs, pad_len, voc_size)

    if hasattr(model, "create_token_phrase_mask"):
        print("yoyo")
        token_phrase_mask = model.create_token_phrase_mask(tokens)
        out = out * token_phrase_mask

    out = torch.log(1 + torch.relu(out)) * tokens["attention_mask"].unsqueeze(-1)

    # out = encode_custom_mask_punc(tokens, model, is_q, False)
    # mask = ~torch.isin(tokens["input_ids"], PUNCID)
    # out = out * mask.unsqueeze(-1)

    return out

tokens = tokenizer(_doc, return_tensors="pt")
out = encode_custom(tokens, model = model, is_q = True)
out.shape


tokens_str = [reverse_voc[int(idx)] for idx in tokens["input_ids"][0]]
row, col = torch.nonzero(out[0][:], as_tuple = True)

token_mapper = [[tokens_str[j], Counter()] for j in range(max(row) + 1)]
for r,c in zip(row, col):
    r_token_id = int(tokens["input_ids"][0][r])
    r_token_str = reverse_voc[r_token_id]

    temp = {}

    c_token_id = int(c)
    c_token_str = reverse_voc[c_token_id]
    if c_token_str not in temp:
        temp[c_token_str] = round(float(out[0][r, c]), 2)

    token_mapper[r][0] = r_token_str
    token_mapper[r][1].update(temp)

In [27]:
for token, counter in token_mapper:
    print(token, "->", Counter(dict(counter)).most_common(50))

[CLS] -> [(',', 1.17), ('the', 1.13), ('david', 0.94), ('son', 0.88), ('company', 0.58), ('.', 0.35), ('fast', 0.33), ('word', 0.31), ('alexander', 0.03)]
un -> [('the', 1.31), (',', 1.07), ('.', 0.83), ('international law', 0.82), ('conflict resolution', 0.74), ('united nations', 0.66), ('un', 0.57), ('mmr', 0.37), ('nicotinamide', 0.33), ('concordance', 0.24), ('conflict management', 0.24), ('free product', 0.23), ('avidin', 0.19), ('bist', 0.18), ('global network', 0.18), ('acknowledgement', 0.17), ('traffic load', 0.16), ('itp', 0.16), ('edc', 0.14), ('low energy', 0.1), ('firewall', 0.07), ('conformance', 0.07), ('skew', 0.06), ('misuse', 0.01), ('pml', 0.01)]
##su -> [('the', 1.11), (',', 1.1), ('.', 0.72), ('david', 0.33), ('itp', 0.14), ('transmission rate', 0.02)]
##per -> [('the', 0.76), (',', 0.65), ('david', 0.45), ('company', 0.37), ('.', 0.19)]
##vis -> [(',', 1.17), ('the', 1.14), ('.', 0.92), ('david', 0.32), ('transmission rate', 0.11)]
##ed -> [(',', 0.86), ('the', 0.